In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
# ----------- Stil für Plots setzen
plt.style.use('seaborn-v0_8-whitegrid')

In [3]:
# ---------- Pfad zu Ordner mit .csv Dateien -----------
base_dir = Path.cwd()

# ---------- Pfad zu "Komplette_Datenbank" in 1.1 Wrapper -----------
db_root = (
    base_dir.parent 
    / "1.1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken"
    / "Komplette_Datenbank"
)

# ---------- Ordner mit neustem Zeitstempel finden -------------
if not db_root.exists():
    raise FileNotFoundError(f"Verzeichnis {db_root} existiert nicht.")

timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Ordner mit Zeitstempel in {db_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
print(f"Neuester Daten-Ordner: {latest_folder.name}")

# --------- Pfad zur .csv Datei --------------
database_path = latest_folder / "Komplette_Datenbank.csv"
if not database_path.exists():
    raise FileNotFoundError(f"Datenbank-Datei nicht gefunden unter {database_path}")

print(f"Datenbank-Pfad: {database_path}")

Neuester Daten-Ordner: 2025-12-30_11-30-56
Datenbank-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Komplette_Datenbank\2025-12-30_11-30-56\Komplette_Datenbank.csv


In [4]:
# ---------- Zielpfad für .pdf -----------
output_pdf = base_dir / "Temperature_Analysis_Report.pdf"
print(f"PDF-Bericht wird gespeichert unter: {output_pdf}")

PDF-Bericht wird gespeichert unter: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2.3_Temperature_Analysis\Temperature_Analysis_Report.pdf


In [5]:
# ---------- Daten laden und Analysieren -----------

# CSV einlesen
print("Lade Datenbank... (das kann einen Moment dauern)")
df = pd.read_csv(database_path, low_memory=False)

# ---------- Temperatur-Spalte identifizieren --------------
temp_col_name = "temperature_in_c"

# ---------- Einzigartige Datenbanken ------------
db_numbers = sorted(df["Database_number"].dropna().unique())

print(f"{len(db_numbers)} einzigartige Datenbanken für die Analyse gefunden")

# ---------- PDF generieren -----------
with PdfPages(output_pdf) as pdf:
    for db_num in db_numbers:
        
        # -------- alle Zeilen dieser Datenbank holen ----------
        df_db = df[df["Database_number"] == db_num]
        db_name = df_db["Database_name"].iloc[0] if "Database_name" in df_db.columns and not df_db["Database_name"].isna().all() else "Unbekannt"
        
        total_rows = len(df_db)
        
        # ---------- Temperatur-Daten und Statistiken ------------
        temp_data = pd.to_numeric(df_db[temp_col_name], errors='coerce')
        temp_not_null = temp_data.dropna()
        count_existing = len(temp_not_null)
        count_missing = total_rows - count_existing
        percent_existing = (count_existing / total_rows * 100) if total_rows > 0 else 0
        percent_missing = (count_missing / total_rows * 100) if total_rows > 0 else 0
        
        # ----------- Statistiken berechnen ------------
        if count_existing > 0:
            t_min = temp_data.min()
            t_max = temp_data.max()
            t_mean = temp_data.mean()
            t_median = temp_data.median()
        else:
            t_min = t_max = t_mean = t_median = np.nan

        # -------------- Text für Seite ---------------
        header_lines = [
            f"Database_number: {db_num}",
            f"Database_name:   {db_name}",
            "",
            f"Anzahl Zeilen insgesamt: {total_rows}",
            f"Werte in '{temp_col_name}': {count_existing} ({percent_existing:0.2f} %)", # Vorhandene Werte angeben
            f"Fehlende Werte: {count_missing} ({percent_missing:0.2f} %)", # Fehlende Werte angeben
            "",
            "Statistik (in °C):",
            f"  Min:    {t_min:0.2f}" if count_existing > 0 else "  Min:    -",
            f"  Max:    {t_max:0.2f}" if count_existing > 0 else "  Max:    -",
            f"  Mittelwert: {t_mean:0.2f}" if count_existing > 0 else "  Mittelwert: -",
            f"  Median: {t_median:0.2f}" if count_existing > 0 else "  Median: -",
        ]

        # ------------ Neue Seite erstellen ------------
        fig = plt.figure(figsize=(8.27, 11.69))  # A4-Hochformat
        
        # ------------ Titel -----------
        fig.suptitle(
            f"Temperatur-Analyse – DB {db_num} ({db_name})",
            fontsize=14,
            fontweight="bold",
            y=0.97
        )
        
        # ------------- Text von oben nach unten platzieren ---------------
        y_start = 0.9
        line_height = 0.025

        for i, line in enumerate(header_lines):
            y = y_start - i * line_height
            fig.text(0.1, y, line, fontsize=10, va="top", family="monospace")
            
        # ------------- Histogramm --------------------
        if count_existing > 0:
            ax = fig.add_axes([0.1, 0.1, 0.8, 0.45]) # [linksseitiger Platz, unterer Platz, Breite, Höhe]
            ax.hist(temp_not_null, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
            ax.set_title("Temperatur-Verteilung", fontsize=12)
            ax.set_xlabel("Temperatur [°C]", fontsize=10)
            ax.set_ylabel("Häufigkeit", fontsize=10)
            
            ax.grid(True, linestyle='--', alpha=0.6)
            ax.axvline(t_mean, color='red', linestyle='dashed', linewidth=1, label=f'Mittelwert: {t_mean:.2f}')
            ax.axvline(t_median, color='green', linestyle='dashed', linewidth=1, label=f'Median: {t_median:.2f}')
            ax.legend(fontsize=9)
        else:
            fig.text(0.5, 0.4, "Keine Temperatur-Daten vorhanden", ha='center', fontsize=12, color='gray')
            plt.axis("off")

        # ------------ als .pdf speichern ---------------
        pdf.savefig(fig)
        plt.close(fig)

print(f"Analyse abgeschlossen. PDF-Bericht erstellt: {output_pdf}")

Lade Datenbank... (das kann einen Moment dauern)


13 einzigartige Datenbanken für die Analyse gefunden


Analyse abgeschlossen. PDF-Bericht erstellt: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2.3_Temperature_Analysis\Temperature_Analysis_Report.pdf
